In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l6.1 RoPE

> 🎯 **Goal:** Replace the learned position table with rotary position embeddings so the model encodes *relative* position and can run past its training length.

**Why this matters.** This is the first upgrade that turns our textbook transformer
into a modern LLM. Earlier units gave each position its own learned vector from a
lookup table capped at `block_size`. That table costs parameters and, worse, the
model has literally never seen a vector for position 5000 if you only trained to
1024, so it cannot generalize beyond its window. Every current frontier model
(LLaMA, Mistral, Qwen) uses RoPE instead.

**The intuition.** Picture each token's query and key as an arrow in space. RoPE
encodes position by *rotating* that arrow by an angle that grows with the token's
position: token 0 is unrotated, token 1 turns a little, token 2 a little more, and
so on. When attention compares a query at position i against a key at position j, the
two have been rotated by different amounts, so what survives the comparison is the
*difference* in their angles, that is, their relative distance. Rotation never
changes an arrow's length, so it injects position without distorting magnitude.

**The mechanics.** A *head* is one of the parallel attention channels; each head
works on a small slice of the embedding called the head size. RoPE pairs up the
dimensions inside that slice and, for each pair, applies a 2D rotation by an angle
`position * frequency`, where low dimensions rotate fast and high dimensions rotate
slowly (a spectrum of frequencies). We precompute the cosines and sines once
(`precompute_rope`) and apply them to q and k just before the dot product
(`apply_rope`). Because the angle is a continuous function of position, positions
the model never trained on still get sensible rotations, which is why RoPE
extrapolates. And because the rotation is fixed math (no weights), it adds zero
*parameters* (the numbers the optimizer learns).

In [ ]:
from pocketlm import precompute_rope, apply_rope, PocketLM, PocketLMConfig

cos, sin = precompute_rope(head_size=8, seq_len=16)
x = torch.randn(1, 2, 5, 8)   # [batch, heads, tokens, head_size]
rotated = apply_rope(x, cos, sin)
print("norm preserved (rotation is length-preserving):",
      torch.allclose(rotated.norm(dim=-1), x.norm(dim=-1), atol=1e-5))
model = PocketLM(PocketLMConfig(vocab_size=20, block_size=16, n_layer=2,
                                n_head=2, n_embd=32, pos="rope"))
print("rope model has a learned pos table:", hasattr(model, "pos_emb"))

**What just happened.** The first check confirms the rotated vectors have the same
per-token norm as the originals: rotation preserved length, exactly as advertised.
The second check shows the rope model has no `pos_emb` attribute, that is, it threw
away the learned position lookup table entirely. Position now lives *inside*
attention, costs no parameters, and works for any sequence length.

⚠️ **Common pitfalls**
- RoPE rotates *queries and keys only*, never the values. Rotating values would
  corrupt the content the model is trying to read out.
- The head size must be even, because RoPE rotates dimensions in pairs. An odd head
  size has a leftover dimension with no partner to rotate against.
- Extrapolation is good but not free: pushing far past the training length still
  degrades quality. Real systems add tricks (position interpolation, NTK scaling) to
  stretch the window further.

🏋️ **Try it yourself.** Call `precompute_rope` with a larger `seq_len` (say 64) and
apply it to a tensor with 64 tokens, then confirm the norm check still passes. You
just ran the model at a length its weights never trained on, and nothing broke.

In [ ]:
assert torch.allclose(rotated.norm(dim=-1), x.norm(dim=-1), atol=1e-5)
assert not hasattr(model, "pos_emb")